## Визуализация train.jsonl

In [ ]:
# интерфейс таблицы
from IPython.display import display, HTML
display(HTML("<style>.jp-Notebook { padding-left: 1% !important; padding-right: 1% !important; width: 98% !important; }</style>"))


In [ ]:
import pandas as pd

# 1. Load the data
df = pd.read_json("/Users/andrewglukhoff/code/chatbot1/data/train.jsonl", lines=True)

# 2. Extract using the exact markers from your example
# We split by the header ID and then take the text before the <|eot_id|>
def extract_medical(text, role):
    marker = f"<|start_header_id|>{role}<|end_header_id|>"
    try:
        # Split by the role marker, take the second part, then split by the end tag
        return text.split(marker)[1].split("<|eot_id|>")[0].strip()
    except:
        return None

df['question'] = df['text'].apply(lambda x: extract_medical(x, 'user'))
df['answer'] = df['text'].apply(lambda x: extract_medical(x, 'assistant'))

# 3. Quick Stats
df['ans_len'] = df['answer'].str.len()
print(f"📊 Total records: {len(df)}")
print(f"📝 Average answer length: {df['ans_len'].mean():.1f} characters")

# 4. Show the results
df[['question', 'answer']].head()


In [ ]:
import pandas as pd

# Убираем ограничение на ширину колонки (чтобы текст не превращался в "...")
pd.set_option('display.max_colwidth', None)

# Увеличим количество отображаемых строк, если нужно
pd.set_option('display.max_rows', 50)

# Выводим таблицу снова
df[['question', 'answer']].head(10)


In [ ]:
# Вывод с выравниванием по левому краю и границами
df[['question', 'answer']].head(10).style.set_properties(**{'text-align': 'left', 'vertical-align': 'top'})


In [ ]:
# Рассчитаем количество слов в ответах
df['word_count'] = df['answer'].str.split().str.len()

print(f"Минимальное кол-во слов: {df['word_count'].min()}")
print(f"Максимальное кол-во слов: {df['word_count'].max()}")
print(f"Медианное кол-во слов: {df['word_count'].median()}") # 11 - очень мало!

# Посмотрим на 5 самых коротких ответов
print("\n--- Самые короткие ответы в базе: ---")
display(df[df['word_count'] > 2].sort_values('word_count').head(5)[['answer', 'word_count']])


In [ ]:
# Посмотрим на 5 самых длинных ответов
print("\n--- Самые длинные ответы в базе: ---")
display(df[df['word_count'] > 2].sort_values('word_count', ascending=False).head(5)[['answer', 'word_count']])

In [ ]:
# Count how many answers are "too short" (e.g., less than 8 words)
short_count = len(df[df['word_count'] < 8])
percent_short = (short_count / len(df)) * 100

print(f"⚠️ Answers with < 8 words: {short_count} ({percent_short:.1f}%)") # 29.7% - много!

# Let's look at what these short answers actually say:
print("\n--- Examples of short answers: ---")
display(df[df['word_count'] < 8]['answer'].head(10))


## Анализ оригинального датасета (~190k)

In [ ]:
import pandas as pd
from datasets import load_dataset

# 1. Загружаем оригинал прямо из Hugging Face (или локального кэша)
# Это эффективнее, чем читать огромный текстовый файл вручную
print("📥 Загрузка полного датасета...")
dataset = load_dataset("blinoff/medical_qa_ru_data", split='train', trust_remote_code=True)

# 2. Превращаем в DataFrame (выбираем только нужные колонки для экономии места)
df_full = pd.DataFrame(dataset)
df_full = df_full[['categ', 'ans']] # Нам нужны только категории и ответы для анализа

In [ ]:
df_full[['categ', 'ans']].head(10)

In [ ]:
# 3. Считаем слова в ответах
df_full['word_count'] = df_full['ans'].astype(str).apply(lambda x: len(x.split()))

# 4. Аналитика
median_words = df_full['word_count'].median()
short_percent = (len(df_full[df_full['word_count'] < 8]) / len(df_full)) * 100

print(f"✅ Готово! Всего записей: {len(df_full)}")
print(f"📊 Медиана слов: {median_words}") # 31 vs 11 in train.jsonl
print(f"⚠️ Процент коротких ответов (< 8 слов): {short_percent:.1f}%") # 9.2% vs 29% in train.jsonl

Why the difference?
Original Median (31 words) vs Your Slice (11 words): Your small sample was unfortunately "unlucky"—it captured a lot of short, conversational junk (greetings, "thank yous", etc.).
The "Gold Mine": With 190k records and only 9% short answers, you have roughly 170,000 high-quality medical dialogues to choose from.
Your Strategy for the "Elite Medical Dataset" 🩺
Since we want a "talkative" and smart doctor, we should create a new train_elite.jsonl. Let's filter for the "best of the best":
Filter by Length: Only answers with word_count > 40 (to ensure detail).
Filter by Category: Keep your favorite areas (Cardiology, etc.).
Shuffle: Take a random 10,000 rows so the model doesn't just learn from one "era" of the forum.

#### формируем другой учебный датасет  (train_elite.json), с учетом анализа предыдущего

In [ ]:
# загрузка ориг/датасета из HF тормозит, поэтому беру из cache
import os
import datasets
import pandas as pd

# 1. Force Offline Mode
os.environ["HF_DATASETS_OFFLINE"] = "1"

print("📥 Loading from local cache (bypassing network)...")
try:
    dataset = datasets.load_dataset(
        "blinoff/medical_qa_ru_data", 
        split='train', 
        trust_remote_code=True
    )
    # 2. Fast conversion to Pandas
    df_full = dataset.to_pandas()
    
    # 3. Quick Word Count
    df_full['word_count'] = df_full['ans'].astype(str).apply(lambda x: len(x.split()))
    
    # 4. Apply your Elite Filter (Word Count >= 40)
    target_cats = ['Терапия', 'Кардиология', 'Неврология', 'Гастроэнтерология', 'Педиатрия']
    df_elite = df_full[
        (df_full['word_count'] >= 40) & 
        (df_full['categ'].isin(target_cats))
    ].copy()

    print(f"✅ Success! Elite dataset rows: {len(df_elite)}")
    
except Exception as e:
    print(f"❌ Error: {e}. If it says file not found, turn off OFFLINE mode once to sync.")


In [ ]:
# 2. Перемешиваем и берем 10,000 лучших строк
df_elite = df_elite.sample(n=10000, random_state=42)

In [ ]:
final_data = []

# Перебираем индексы нашего отфильтрованного df_elite
for i in df_elite.index:
    # 1. Извлекаем вопрос и ответ напрямую из объекта dataset
    q = dataset[int(i)]['desc'].strip()
    
    # 2. Чистим ответ: заменяем ';' на пробел и убираем лишние пробелы по краям
    a = dataset[int(i)]['ans'].replace(';', ' ').strip()
    
    # 3. Форматируем под Llama-3 (без лишних \n\n, как в твоем рабочем примере)
    entry = {
        "text": f"<|begin_of_text|><|start_header_id|>user<|end_header_id|> {q}<|eot_id|><|start_header_id|>assistant<|end_header_id|> {a}<|eot_id|>"
    }
    final_data.append(entry)

# 4. Сохранение
import json
with open("/Users/andrewglukhoff/code/chatbot1/data/train_elite.jsonl", "w", encoding="utf-8") as f:
    for item in final_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"✅ Готово! Создан файл 'train_elite.jsonl' на {len(final_data)} строк.")
